# Week 1: train ONE ensemble member (60 classes, n_fft 64)

Each teammate runs this once with **their own seed**. Do not change the seed or any setting after you start.

| Seed | Who |
|---|---|
| 1, 2 | Eavan |
| 3 | Chua |
| 4 | Jessy |
| 5 | Eileen |

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

**If Colab disconnects:** just run every cell again from the top with the same seed. Training resumes from the last finished epoch (`last.pt` is saved to Drive after every epoch).

## 1. Settings: change only `SEED` (and `FOI26` if your Drive path is different)

In [ ]:
SEED = 1          # <-- your seed: Eavan 1 and 2, Chua 3, Jessy 4, Eileen 5
BRANCH = 'eavan-merge-shards'

# The shared FOI26 folder in Drive. If it was shared WITH you, open it in Drive,
# right-click -> Organise -> Add shortcut -> My Drive, so it appears here.
FOI26 = '/content/drive/MyDrive/sedic/FOI26'

DATA_ON_DRIVE = f'{FOI26}/combined_data_new'       # train/ val/ test/ + combined.json (3fae376e92d3: external fix)
# One folder per run inside results/: training date + what is being tried + seed.
# Type the date by hand (not today's date from code): if Colab disconnects and you
# rerun tomorrow, the SAME name is what lets training resume instead of restarting.
RUN_NAME = '2026-11-05_nfft64_ep15_extfix'   # <-- date_feature; change it only for a NEW experiment
EPOCHS = 15                           # 15 = first check (~1-1.5 h); 30 = the real ensemble run
CKPT_DIR      = f'{FOI26}/results/{RUN_NAME}_seed{SEED}'  # last.pt, best.pt, history.json, scorecard.json
# Only for the real ensemble run: then set it to f'{FOI26}/members/seed{SEED}.pt'.
# Left empty for a first check, so a trial model never lands in members/.
MEMBER_OUT    = ''

## 2. GPU check

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only: change the runtime to T4 GPU first')

## 3. Mount Drive and get the code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content
!rm -rf sedicAI_NEXA
!git clone -q -b {BRANCH} https://github.com/eavan127/sedicAI_NEXA.git
%cd /content/sedicAI_NEXA
!git log -1 --format='commit %H  %s'
!pip install -q pyyaml

## 4. Copy the data to the Colab disk

Reading straight from Drive during training is very slow, so copy it once (~2.4 GB, a few minutes).
The check below stops here if the copy is incomplete.

In [ ]:
import json, os, numpy as np
!mkdir -p /content/combined
!cp -r "{DATA_ON_DRIVE}/train" "{DATA_ON_DRIVE}/val" "{DATA_ON_DRIVE}/test" /content/combined/
!cp "{DATA_ON_DRIVE}/combined.json" /content/combined/ || echo 'combined.json missing on Drive: upload it from data/combined/'

for split, n in (('train', 960_000), ('val', 120_000), ('test', 120_000)):
    X = np.load(f'/content/combined/{split}/X.npy', mmap_mode='r')
    y = np.load(f'/content/combined/{split}/y.npy', mmap_mode='r')
    print(split, X.shape, X.dtype, y.shape, y.dtype)
    assert X.shape == (n, 2, 512) and X.dtype in (np.float16, np.float32) and y.shape == (n, 60), f'{split} looks wrong: re-upload it'
if os.path.exists('/content/combined/combined.json'):
    print('combined version:', json.load(open('/content/combined/combined.json'))['combined_version'], '(expect de26671a5d48)')

## 5. Quick check (about 1 minute)

Trains on 2,000 windows for 1 epoch into a throwaway folder, just to prove everything runs. **If this fails, stop and send the error to Eavan.**

In [ ]:
!python scripts/train_week1.py --seed {SEED} --data /content/combined --limit 2000 --epochs 1 --batch-size 256 --ckpt-dir /content/try --fresh

## 6. Train (about 1-1.5 h for 15 epochs, 2-3 h for 30, on a T4)

Prints one line per epoch with the time left. Every epoch is saved to `CKPT_DIR` (`results/<RUN_NAME>_seed<k>/`) on Drive.
When it finishes it scores the test split (`scorecard.json`) and, if `MEMBER_OUT` is set, copies the best model there.

**Write in your run log:** RUN_NAME, seed, the commit printed in step 3, the combined version printed in step 4.


In [ ]:
member = f'--member-out "{MEMBER_OUT}"' if MEMBER_OUT else ''
!python scripts/train_week1.py --seed {SEED} --data /content/combined --epochs {EPOCHS} --batch-size 256 --ckpt-dir "{CKPT_DIR}" {member} --eval-test


## 7. Results

In [ ]:
h = json.load(open(f'{CKPT_DIR}/history.json'))
print('best epoch:', h['best_epoch'])
for e in h['epochs']:
    print(f"{e['epoch']:>2}  val_loss {e['val_loss']:.4f}  top1 {e['val_top1']:.3f}  recall@0.5 {e['val_macro_recall_05']:.3f}  {e['seconds']:.0f}s")
if os.path.exists(f'{CKPT_DIR}/scorecard.json'):
    t = json.load(open(f'{CKPT_DIR}/scorecard.json'))['test']
    print()
    print(f"TEST  top1 {t['top1']:.3f}  macro recall@0.5 {t['macro_recall_05']:.3f}")
    worst = sorted(t['per_class'].items(), key=lambda kv: kv[1]['top1_standalone'])[:10]
    print('10 weakest classes (top-1):', [(c, round(v['top1_standalone'], 2)) for c, v in worst])

## 8. Detailed report (about 2 minutes)

Runs `best.pt` on the test split: accuracy by SNR, confusion matrices (all SNRs and >= 10 dB), class x SNR heatmap, per-class table with what each class is mistaken for. Charts and `per_class.csv` are saved to `CKPT_DIR/report/`. Needs steps 1, 3 and 4 in this session.

In [ ]:
# ===== Evaluation report: SNR curve + confusion matrices + per-class table =====
import sys, json, os, numpy as np, torch, pandas as pd, matplotlib.pyplot as plt
sys.path.insert(0, '/content/sedicAI_NEXA'); sys.path.insert(0, '/content/sedicAI_NEXA/scripts')
from src.config import CFG, DATASET_CLASSES
from src.models.amc_cnn import AMC_CNN
from train_week1 import load_split

DATA, HIGH_SNR = '/content/combined', 10      # windows at >= HIGH_SNR dB count as "clean"
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
C, CL = len(DATASET_CLASSES), DATASET_CLASSES
ck = torch.load(f'{CKPT_DIR}/best.pt', map_location=dev, weights_only=False)
model = AMC_CNN(num_classes=C, input_len=ck['meta']['input_len'], stft_n_fft=ck['meta']['stft_n_fft'], **ck['meta']['model_flags']).to(dev)
model.load_state_dict(ck['state_dict']); model.eval()
X, y, snr = load_split(DATA, 'test')
logits = np.empty((len(X), C), np.float32)
with torch.no_grad():
    for s in range(0, len(X), 1024):
        logits[s:s+1024] = model(torch.from_numpy(X[s:s+1024]).to(dev).float()).cpu().numpy()

yt = y.astype(bool); top = logits.argmax(1); hit = yt[np.arange(len(yt)), top]
single = yt.sum(1) == 1; true = yt.argmax(1); high = snr >= HIGH_SNR
print(f"TEST top1 {hit.mean():.3f} | single-signal {hit[single].mean():.3f} | mixtures {hit[~single].mean():.3f} | at >= {HIGH_SNR} dB {hit[high].mean():.3f}")

# 1) accuracy by SNR
bins = CFG['dataset_snr_bins_db']
snr_tab = pd.DataFrame([{'snr_db': b, 'all': hit[snr == b].mean(), 'single': hit[(snr == b) & single].mean(),
                         'mixtures': hit[(snr == b) & ~single].mean(), 'windows': int((snr == b).sum())} for b in bins])
print(snr_tab.round(3).to_string(index=False))
ax = snr_tab.plot(x='snr_db', y=['all', 'single', 'mixtures'], marker='o', figsize=(8, 4), ylim=(0, 1), grid=True, title='Test top-1 by SNR')
ax.set_xlabel('SNR (dB)'); ax.set_ylabel('top-1')
os.makedirs(f'{CKPT_DIR}/report', exist_ok=True); plt.savefig(f'{CKPT_DIR}/report/snr_curve.png', dpi=130); plt.show()

# 2) confusion matrices (single-signal windows; row = true class, each row sums to 1)
def conf(mask):
    m = np.zeros((C, C)); np.add.at(m, (true[mask], top[mask]), 1)
    return m, m / np.maximum(m.sum(1, keepdims=True), 1)
def heat(m, title, fname, xl=CL, size=(15, 14), xlabel='predicted'):
    fig, ax = plt.subplots(figsize=size); im = ax.imshow(np.nan_to_num(m), cmap='Blues', vmin=0, vmax=1, aspect='auto')
    ax.set_xticks(range(len(xl))); ax.set_xticklabels(xl, rotation=90, fontsize=6)
    ax.set_yticks(range(C)); ax.set_yticklabels(CL, fontsize=6); ax.set_xlabel(xlabel); ax.set_ylabel('true'); ax.set_title(title)
    fig.colorbar(im, fraction=0.03); fig.tight_layout(); fig.savefig(f'{CKPT_DIR}/report/{fname}', dpi=130); plt.show()
cnt_all, cm_all = conf(single); _, cm_high = conf(single & high)
heat(cm_all, 'Confusion, all SNRs', 'confusion_all.png')
heat(cm_high, f'Confusion, SNR >= {HIGH_SNR} dB (mistakes here are real confusions, not noise)', 'confusion_high.png')

# 3) per-class top-1 at every SNR
grid = np.array([[hit[single & (true == i) & (snr == b)].mean() if (single & (true == i) & (snr == b)).any() else np.nan
                  for b in bins] for i in range(C)])
heat(grid, 'Top-1 per class and SNR', 'class_by_snr.png', xl=[str(b) for b in bins], size=(8, 14), xlabel='SNR (dB)')

# 4) per-class table + most common mistakes
rows = []
for i, c in enumerate(CL):
    mine = single & (true == i); r = cnt_all[i].copy(); r[i] = 0; j = r.argmax()
    tp = (logits[:, i] > 0) & yt[:, i]
    rows.append({'class': c, 'windows': int(mine.sum()), 'top1': hit[mine].mean(), f'top1_>={HIGH_SNR}dB': hit[mine & high].mean(),
                 'recall@0.5': tp.sum() / max(yt[:, i].sum(), 1), 'precision@0.5': tp.sum() / max((logits[:, i] > 0).sum(), 1),
                 'mistaken_for': CL[j], 'mistake_share': r[j] / max(cnt_all[i].sum(), 1)})
per_class = pd.DataFrame(rows).sort_values('top1')
per_class.round(3).to_csv(f'{CKPT_DIR}/report/per_class.csv', index=False)
pd.set_option('display.width', 200)
print('\nPER CLASS (worst first):'); print(per_class.round(2).to_string(index=False))
pairs = sorted(((cm_all[i, j], CL[i], CL[j]) for i in range(C) for j in range(C) if i != j), reverse=True)[:15]
print('\nMOST COMMON MISTAKES (true -> predicted, share of that class):')
for s, a, b in pairs: print(f'  {a:>14} -> {b:<14} {s:.2f}')
print(f'\nSaved to {CKPT_DIR}/report/')

## 9. Accuracy per signal family

`exact_class`: named the exact class. `right_family`: at least the right family (e.g. said 8PSK for 16PSK). Run after step 8.

In [ ]:
# ===== Accuracy per signal family (run after the evaluation cell) =====
from src.config import CLASS_GROUPS
fam = {c: g for g, cs in CLASS_GROUPS.items() for c in cs}; fam['NOISE_FLOOR'] = 'NOISE'
tf = np.array([fam[CL[i]] for i in true]); pf = np.array([fam[CL[i]] for i in top])
fam_tab = pd.DataFrame([{'family': g, 'classes': sum(f == g for f in fam.values()), 'windows': int((single & (tf == g)).sum()),
    'exact_class': hit[single & (tf == g)].mean(), 'exact_>=10dB': hit[single & high & (tf == g)].mean(),
    'right_family': (pf == tf)[single & (tf == g)].mean(), 'right_family_>=10dB': (pf == tf)[single & high & (tf == g)].mean()}
    for g in sorted(set(fam.values()))]).sort_values('exact_class')
print(fam_tab.round(3).to_string(index=False))
fam_tab.round(3).to_csv(f'{CKPT_DIR}/report/per_family.csv', index=False)

## 10. "No signal" threshold T

If the best class score is below T, answer NOISE_FLOOR. Pick the highest T where *signals wrongly called noise (>=10dB)* stays at about 2-3% or less. Seed 1 (2026-11-04): T = 0.1 took noise from 3.9% to 50% for a 0.2-point loss on clean signals. Run after step 8.

In [ ]:
# ===== Test a "no confident signal -> NOISE" rule on the current model (no retraining) =====
p = 1 / (1 + np.exp(-np.clip(logits, -50, 50))); best = p.max(1); NI = CL.index('NOISE_FLOOR')
isn = yt[:, NI]; sig = ~isn
print(' T    noise found | signal top1 (all / >=10dB) | signals wrongly called noise (all / >=10dB)')
for T in [0, 0.05, 0.08, 0.10, 0.12, 0.14, 0.16, 0.18, 0.20]:
    pred = np.where(best < T, NI, top); h = yt[np.arange(len(yt)), pred]; calln = best < T
    print(f'{T:.1f}     {h[isn].mean():.3f}     |     {h[sig].mean():.3f} / {h[sig & high].mean():.3f}       |     {calln[sig].mean():.3f} / {calln[sig & high].mean():.3f}')